Evan Edelstein
EN.605.645.82.SP26

# Module 8 - Programming Assignment

## Directions

1. Change the name of this file to be your JHED id as in `jsmith299.ipynb`. Because sure you use your JHED ID (it's made out of your name and not your student id which is just letters and numbers).
2. Make sure the notebook you submit is cleanly and fully executed. I do not grade unexecuted notebooks.
3. Submit your notebook back in Blackboard where you downloaded this file.

*Provide the output **exactly** as requested*

In [1]:
import json
import random
from copy import deepcopy
from math import inf, log2
from typing import Dict, List, NamedTuple, Tuple, Callable, Set

## Decision Trees

For this assignment you will be implementing and evaluating a Decision Tree using the ID3 Algorithm (**no** pruning or normalized information gain). Use the provided pseudocode. The data is located at (copy link):

http://archive.ics.uci.edu/ml/datasets/Mushroom

**Just in case** the UCI repository is down, which happens from time to time, I have included the data and name files on Canvas.

<div style="background: lemonchiffon; margin:20px; padding: 20px;">
    <strong>Important</strong>
    <p>
        No Pandas. The only acceptable libraries in this class are those contained in the `environment.yml`. No OOP, either. You can used Dicts, NamedTuples, etc. as your abstract data type (ADT) for the the tree and nodes.
    </p>
</div>

One of the things we did not talk about in the lectures was how to deal with missing values. There are two aspects of the problem here. What do we do with missing values in the training data? What do we do with missing values when doing classifcation?

There are a lot of different ways that we can handle this.
A common algorithm is to use something like kNN to impute the missing values.
We can use conditional probability as well.
There are also clever modifications to the Decision Tree algorithm itself that one can make.

We're going to do something simpler, given the size of the data set: remove the observations with missing values ("?").

You must implement the following functions:

`train` takes training_data and returns the Decision Tree as a data structure.

```
def train(training_data):
   # returns the Decision Tree.
```

`classify` takes a tree produced from the function above and applies it to labeled data (like the test set) or unlabeled data (like some new data).

```
def classify(tree, observations):
    # returns a list of classifications
```

`evaluate` takes a data set with labels (like the training set or test set) and the classification result and calculates the classification error rate:

$$error\_rate=\frac{errors}{n}$$

Do not use anything else as evaluation metric or the submission will be deemed incomplete, ie, an "F". (Hint: accuracy rate is not the error rate!).

`cross_validate` takes the data and uses 10 fold cross validation (from Module 3!) to `train`, `classify`, and `evaluate`. **Remember to shuffle your data before you create your folds**. I leave the exact signature of `cross_validate` to you but you should write it so that you can use it with *any* `classify` function of the same form (using higher order functions and partial application).

Following Module 3's material (course notes), `cross_validate` should print out a table in exactly the same format. What you are looking for here is a consistent evaluation metric cross the folds. Print the error rate to 4 decimal places. **Do not convert to a percentage.**

```
def pretty_print_tree(tree):
    # pretty prints the tree
```

This should be a text representation of a decision tree trained on the entire data set (no train/test).

To summarize...

Apply the Decision Tree algorithm to the Mushroom data set using 10 fold cross validation and the error rate as the evaluation metric. When you are done, apply the Decision Tree algorithm to the entire data set and print out the resulting tree.

**Note** Because this assignment has a natural recursive implementation, you should consider using `deepcopy` at the appropriate places.


### Provided Functions

You do not need to document these.

You can use this function to read the data file.

In [2]:
def parse_data(file_name: str) -> list[list]:
    data = []
    file = open(file_name, "r")
    for line in file:
        datum = line.rstrip().split(",")
        data.append(datum)
    random.shuffle(data)
    return data

You can use this function to create 10 folds for 5x2 cross validation.

In [3]:
def create_folds(xs: list, n: int) -> list[list[list]]:
    k, m = divmod(len(xs), n)
    # be careful of generators...
    return list(xs[i * k + min(i, m) : (i + 1) * k + min(i + 1, m)] for i in range(n))

Put your code after this line:

-----

# Tree

In [4]:
Node = NamedTuple("Node", [("value", str), ("children", List)])

<a id="create_node"></a>
## create_node

*`create_node` create a node in a tree, each node is a NamedTuples with two attributes, "value" which is the feature being partitioned by (for an internal node) or label value (leaf node) and "children" which is a list of tuples, each holding the edge value (attribute) and child node.* **Used by**: [get_leaf_node](#get_leaf_node) and [id3](#id3)

* **value** str - attribute  that the node represents a partitioned by or if the node is a leaf, the label value

**returns** Node: node in a tree 

In [5]:
def create_node(value: str) -> Node:
    return Node(value, [])

In [6]:
node = create_node("foo")
assert node.value == "foo"  # test 1 - has node
assert node.children == []  # test 2 - has children attribute

node = create_node("")
assert node.value == ""  # test 3 empty list

<a id="add_child"></a>
## add_child

*`add_child` add a child node to a parent node with a specified edge value (attribute). If the edge value is already present in the children of the parent, the child is not added* **Used by**: [id3](#id3)

* **parent** Node - parent node, whose value is a feature
* **child** Node - child node, whose value is either a feature or a label
* **edge_value** str - attribute being partitioned on, represented as a label on the edge from parent to child

**returns** Node: possibly modified parent node

In [7]:
def add_child(parent: Node, child: Node, edge_value: str) -> Node:
    if any(edge_value == v for v, _ in parent.children):
        return parent

    parent.children.append((edge_value, child))
    return parent

In [8]:
parent = create_node("child")
child1 = create_node("child1")
child2 = create_node("child2")

parent = add_child(parent, child1, "edge1")
assert parent.children == [("edge1", child1)]  # test 1 - add child node and edge to tree

parent = add_child(parent, child2, "edge2")
assert parent.children == [("edge1", child1), ("edge2", child2)]  # test 2 - add child node and edge to children

parent = add_child(parent, child1, "edge2")
assert parent.children == [("edge1", child1), ("edge2", child2)]  # test 3 - dont add child if already in edge already taken

<a id="pretty_print_tree"></a>
## pretty_print_tree

*`pretty_print_tree` using DFS, print the paths through a decision tree, with indentation to denote node depth. There are three cases to handle. First a partitioning point is displayed by printing the nodes value (feature). Next, each child is iterated through. If the child is a leaf node, the attribute being selected and the label of the leaf is printed. If the child is an internal node, just the attribute being selected is printed.* **Used by**: [run_model](#run_model)

* **node** Node - root of the tree
* **indent** int - indentation level to print each current nodes depth at. 

**returns** 

In [9]:
def pretty_print_tree(node: Node, indent: int = 0):
    tabs = " " * indent
    print(f"{tabs} - {node.value}")  # parent (attr to be split)
    for attribute, child in node.children:
        tabs = " " * (indent + 4)
        if not child.children:
            print(f"{tabs} | {attribute} -> {child.value}")  # leaf node - value is label
        else:
            print(f"{tabs} | {attribute}")  # internal node - just print partitioned attribute
            pretty_print_tree(child, indent + 8)
    return

In [10]:
tree = Node(value="a", children=[("1", Node(value="b", children=[]))])

print("test 1")
pretty_print_tree(tree)  # test 1 - one child leaf

print("test 2")

tree = Node("", [])
pretty_print_tree(tree)  # test 2 - empty list

print("test 3")

tree = Node(
    "1",
    [
        ("a", Node("y", [])),
        ("b", Node("2", [("c", Node("n", []))])),
    ],
)
pretty_print_tree(tree)  # test 3 - multiple children

test 1
 - a
     | 1 -> b
test 2
 - 
test 3
 - 1
     | a -> y
     | b
         - 2
             | c -> n


<a id="traverse_tree"></a>
## traverse_tree

*`traverse_tree` using DFS, traverse a tree starting from node and follow the edges from a list of observations. For each node, we get its feature by inspecting the nodes value. The column number corresponding to the feature is looked up from the feature_indices dict. If there is an edge from the parent whose value (attribute) matches the corresponding value in the observation list, that edge is taken until a leaf node is hit, then the nodes value (label) is returned. If the tree cannot be traversed None is returned. Using this method, we can traverse a decision tree to produce an label estimate based on a list of observed attributes* **Used by**: [classify](#classify)

* **node** Node - root node of the tree
* **observation** List[str] - list of attributes to find label of 
* **feature_indices** Dict[int, str] - map of positions in observation to each feature in the tree

**returns** str | None - estimate label for the observation or None is tree cannot be traversed

In [11]:
def traverse_tree(node: Node, observation: List[str], feature_indices: Dict[str, int]) -> str | None:
    if len(node.children) == 0:
        return node.value  # label since node is leaf

    feature = node.value  # feature since node is not leaf
    if feature not in feature_indices:
        return None

    column_idx = feature_indices[feature]
    observed_attr = observation[column_idx]
    for attribute, child in node.children:
        if attribute == observed_attr:
            return traverse_tree(child, observation, feature_indices)
    return None

In [12]:
tree = Node("1", [("a", Node("y", [])), ("b", Node("n", []))])

assert traverse_tree(tree, ["a"], {"1": 0}) == "y"  # test 1 - observation in tree returns leaf label
assert traverse_tree(tree, ["c"], {"1": 0}) is None  # test 2 - observation not in tree return None

tree = Node("1", [("a", Node("2", [("b", Node("y", [])), ("c", Node("n", []))]))])
assert traverse_tree(tree, ["a", "b"], {"1": 0, "2": 1}) == "y"  # test 3 - search through tree

# I/O and Data Parsing

<a id="parse_attributes"></a>
## parse_attributes

*`parse_attributes` parse an attributes json file given by filename. The file contains a mapping of each feature to a nested map of encoding of the attribute in the data, to the full name of the attribute to be displayed in the tree. Two dictionaries are returned. The first is a map of each feature to a tuple containing the index of that feature in the dataset and the list of attributes in the domain of the feature. The second maps each feature and encoded attribute to the full name of the attribute. Note, the label and its domain should be included in the json file. The order of each feature in the json should match the order they appear in each row of the data set.*

* **filename** str - filepath to a json of features and attributes - The order of each feature should match the order they appear in the data set.


**returns** Tuple[Dict[str, Tuple[int, List[str]]], Dict[str, Dict[str. str]]] - a map of each feature (and label) to a tuple with its position in the dataset and a list of attributes in the domain of the feature, and a nested map of each feature and encoded attribute to the full name of the attribute

In [13]:
def parse_attributes(filename: str) -> Tuple[Dict[str, Tuple[int, List[str]]], Dict[str, Dict[str, str]]]:
    abrv2fullname: Dict[str, Dict[str, str]] = {}
    attributes: Dict[str, Tuple[int, List[str]]] = {}

    with open(filename, "r") as fh:
        data: Dict[str, Dict[str, str]] = json.load(fh)

    for idx, (feature, attrs) in enumerate(data.items()):
        abrv2fullname[feature] = {}
        attributes[feature] = (idx, [])
        for name, code in attrs.items():
            attributes[feature][1].append(name)
            abrv2fullname[feature][code] = name

    return attributes, abrv2fullname

In [14]:
filename = "./agaricus-lepiota-3.attrs.json"
attributes, abrv2fullname = parse_attributes(filename)

attribute_keys = [
    "mushroom-type",
    "cap-shape",
    "cap-surface",
    "cap-color",
    "bruises?",
    "odor",
    "gill-attachment",
    "gill-spacing",
    "gill-size",
    "gill-color",
    "stalk-shape",
    "stalk-root",
    "stalk-surface-above-ring",
    "stalk-surface-below-ring",
    "stalk-color-above-ring",
    "stalk-color-below-ring",
    "veil-type",
    "veil-color",
    "ring-number",
    "ring-type",
    "spore-print-color",
    "population",
    "habitat",
]

assert list(attributes.keys()) == attribute_keys  # test 1 - all keys are present
assert all([len(a) > 1 for a in attr] for _, attr in attributes.values())  # test 2 - all attribute names are full name
assert all([len(k) == 1 and len(v) > 0 for k, v in a.items()] for a in abrv2fullname.values())  # test 3 - can map from single letter to full name

<a id="rename_data"></a>
## rename_data

*`rename_data` Convert all values in data from an encoded attribute to the full attribute name given by the nested dictionary abrv2name, which maps each feature to a dictionary of encoded attribute values to full attribute name. A dictionary that maps each feature to its domain is also required. If a value in data cannot be translated, None is returned.*

* **data** List[List[str]] - a 2d list of observed attributes.
* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (and label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **abrv2name** Dict[str, Dict[str. str]] - a nested map of each feature and encoded attribute to the full name of the attribute


**returns** List[List[str]] | None - a copy of data with all values translated to their full name or None if a value cannot be translated

In [15]:
def rename_data(data: List[List[str]], attributes: Dict[str, Tuple[int, List[str]]], abrv2name: Dict[str, Dict[str, str]]) -> List[List[str]] | None:
    new_data = []
    for row in data:
        if len(row) != len(attributes):
            return None

        new_row = []
        for value, attr in zip(row, attributes):
            if attr in abrv2name and value in abrv2name[attr]:
                new_row.append(abrv2name[attr][value])
            else:
                return None
        new_data.append(new_row)

    return new_data

In [16]:
data = [["a", "b", "c"], ["a", "b", "c"]]
attributes = {"1": (0, ["a"]), "2": (1, ["b"]), "3": (2, ["c"])}
abrv2name = {"1": {"a": "aaa"}, "2": {"b": "bbb"}, "3": {"c": "ccc"}}

result = rename_data(data, attributes, abrv2name)
assert result is not None and result[0][0] == "aaa" and result[0][1] == "bbb" and result[0][2] == "ccc" and result[1][0] == "aaa" and result[1][1] == "bbb" and result[1][2] == "ccc"  # test 1 - normal replacement


data = [["a", "b", "c"]]
attributes = {"1": (0, ["a"]), "2": (1, ["b"]), "3": (2, ["c"])}
abrv2name = {"2": {"b": "bbb"}, "3": {"c": "ccc"}}

result = rename_data(data, attributes, abrv2name)
assert result is None  # test 2 - missing attribute in map


data = [["a", "b", "c", "d"]]
attributes = {"1": (0, ["a"]), "2": (1, ["b"]), "3": (2, ["c"])}
abrv2name = {"1": {"a": "aaa"}, "2": {"b": "bbb"}, "3": {"c": "ccc"}}
result = rename_data(data, attributes, abrv2name)
assert result is None  # test 3 - extra value in data

<a id="split_features"></a>
## split_features

*`split_features` Given a mapping where each key is a feature, extract all the keys except the one matching label.* **Used by**: [run_model](#run_model)

* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (or label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **label** str - key to skip when scanning attributes


**returns** Set[str] - a set of all non-label features 

In [17]:
def split_features(attributes: Dict[str, Tuple[int, List[str]]], label: str) -> Set[str]:
    return {i for i in attributes if i != label}

In [18]:
attributes = {"1": (0, ["a"]), "2": (1, ["b"]), "3": (2, ["c"])}
label = "3"
features = split_features(attributes, label)
assert features == {"1", "2"}  # test 1 - splits features and labels

attributes = {"1": (0, ["a"]), "2": (1, ["b"]), "3": (2, ["c"])}
label = "4"
features = split_features(attributes, label)
assert features == {"1", "2", "3"}  # test 2 - label doesn't exist in attributes


attributes = {"1": (0, ["a"])}
label = "1"
features = split_features(attributes, label)
assert features == set()  # test 3 - only label so features is empty


# ID3

<a id="get_homogeneous_label"></a>
## get_homogeneous_label

*`get_homogeneous_label` Given a 2d list of data values, check if the label column contains the same value over all rows and return the label. If the rows contain more than one label, the label index is out of bounds or the data is empty, None is returned* **Used by**: get_leaf_node[](#get_leaf_node)

* **data** List[List[str]] - a 2d list of observed attributes and label.
* **label_idx** int - position of label value in a row


**returns** Str | None - the homogenous label if it exists else None

In [19]:
def get_homogeneous_label(data: List[List[str]], label_idx: int) -> str | None:
    if len(data) == 0:
        return None

    if not all(len(row) and 0 <= label_idx < len(row) for row in data):
        return None

    if len(set(row[label_idx] for row in data)) == 1:
        return data[0][label_idx]
    return None

In [20]:
data = [["a", "b", "c"], ["a", "b", "c"]]
assert get_homogeneous_label(data, 0) == "a"  # test 1 - column is the same

data = [["a", "b"], ["b", "c"]]
assert get_homogeneous_label(data, 1) is None  # test 2 - column is different

data = [["a", "b", "c"], ["a", "b", "c"]]
assert get_homogeneous_label(data, 4) is None  # test 3 - out of bounds label index

data = [[], []]
assert get_homogeneous_label(data, 0) is None  # test 3 - empty lists return None

<a id="get_majority_label"></a>
## get_majority_label

*`get_majority_label` Given a 2d list of data values and the index of the label column, get the majority label value. if the label index is out of bounds or the data is empty, None is returned* **Used by**: [get_leaf_node](#get_leaf_node), [id3](#id3)

* **data** List[List[str]] - a 2d list of observed attributes and label.
* **label_idx** int - position of label value in row


**returns** str | None - majority label value in the data or None if label index is out of bounds or data is empty

In [21]:
def get_majority_label(data: List[List[str]], label_idx: int, trace: bool = False) -> str | None:
    if not data or not all(0 <= label_idx < len(row) for row in data):
        return None

    counts = {}
    for row in data:
        label = row[label_idx]
        if label not in counts:
            counts[label] = 1
        else:
            counts[label] += 1

    label = max(counts, key=lambda k: counts[k])

    print(f"Majority label: {label}") if trace else None
    return label

In [22]:
data = [["a", "b", "c"], ["a", "b", "c"], ["c", "b", "a"]]
assert get_majority_label(data, 0) == "a"  # test 1 - get first label

data = [["a", "b", "c"], ["a", "b", "c"]]
assert get_majority_label(data, 4) is None  # test 2 - out of bounds label index

data = [[]]
assert get_majority_label(data, 0) is None  # test 3 - empty lists returns None

<a id="calculate_entropy"></a>
## calculate_entropy

*`calculate_entropy` Given a 2d list of data values, calculate the entropy for a given feature. The entropy per attribute of a feature is given by: $$E(S_a) = -\sum_{l \in L }p_l log2(p_l)$$ where $L$ is the domain of label values, $S_a$ is the subset of the data with the attribute $a$ and $p_l$ is the occurrences of rows in $S_a$ with label l divided by the size of $S_a$. The weighted entropy is computed by: $$\sum_{a \in f}\frac{|S_a|}{|S|}E(S_a)$$, where $a \in f$ is the attribute domain for feature $f$ and $|S|$ is the size of input data set. The label name and a map of each feature (and label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature is required.* **Used by**: [pick_best_feature](#pick_best_feature)

* **data** List[List[str]] - a 2d list of observed attributes and label
* **feature** str - feature to calculate entropy of
* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (or label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **label** str - label name


**returns** float - weighted entropy of feature or None if entropy cannot be calculated

In [ ]:
def calculate_entropy(data: List[List[str]], feature: str, attributes: Dict[str, Tuple[int, List[str]]], label: str) -> float | None:
    if feature not in attributes or label not in attributes:
        return None
    
    total_entropy, total_size = 0, len(data)
    feature_idx, attribute_list = attributes[feature]
    label_idx, label_list = attributes[label]
    
    for attribute in attribute_list:
        s_a = [row for row in data if row[feature_idx] == attribute]
        if len(s_a) == 0:
            continue
        entropy = 0
        for label_value in label_list:
            p_l = len([row for row in s_a if row[label_idx] == label_value]) / len(s_a)
            if p_l <= 0.0:
                continue
            entropy += -1 * p_l * log2(p_l)
        total_entropy += (len(s_a) / total_size) * entropy
    return total_entropy

In [24]:
data = [["a", "y"], ["a", "y"], ["b", "n"], ["b", "n"]]
attributes = {"1": (0, ["a", "b"]), "2": (1, ["y", "n"])}
assert calculate_entropy(data, "1", attributes, "2") == 0  # test 1 - perfect split has entropy of 0

data = [["a", "y"], ["b", "y"], ["a", "n"], ["b", "n"]]
assert calculate_entropy(data, "1", attributes, "2") == 1  # test 1 - 50/50 split has entropy of 1

data = [["a", "y"], ["b", "y"], ["a", "n"], ["b", "n"]]
attributes = {"1": (0, ["a", "b"]), "2": (1, ["y", "n"])}
assert calculate_entropy(data, "3", attributes, "2") is None  # test 1 - missing attribute returns None

<a id="pick_best_feature"></a>
## pick_best_feature

*`pick_best_feature` Given a 2d list of data values and a set of features, find the feature with the lowest entropy. The label name and a map of each feature (and label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature is required.* **Uses** [calculate_entropy](#calculate_entropy) **Used by**: [id3](#id3)

* **data** List[List[str]] - a 2d list of observed attributes and label
* **feature** Set[str] - features to calculate gain of
* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (or label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **label** str - label name


**returns** str - feature with the lowest entropy

In [25]:
def pick_best_feature(data: List[List[str]], features: Set[str], attributes: Dict[str, Tuple[int, List[str]]], label: str, trace: bool = False) -> str | None:
    best_feature, best_entropy = None, inf

    for feature in features:
        entropy = calculate_entropy(data, feature, attributes, label)
        if entropy is None:
            print(f"entropy of feature {feature} is None") if trace else None
            continue
        print(f"entropy of feature {feature} = {entropy:.3f}") if trace else None

        if entropy < best_entropy:
            best_feature = feature
            best_entropy = entropy

    if trace:
        print(f"Highest entropy feature {best_feature} = {best_entropy:.3f}") if best_feature else print("Highest entropy feature not found")
    return best_feature

In [26]:
data = [["a", "c", "y"], ["a", "d", "y"], ["b", "c", "n"], ["b", "d", "n"]]
attributes = {"1": (0, ["a", "b"]), "2": (1, ["c", "d"]), "3": (2, ["y", "n"])}
label = "3"
features = {"1", "2"}


feature = pick_best_feature(data, features, attributes, label)
assert feature == "1"  # test 1 - first column splits data


data = [["a", "c", "n"], ["b", "c", "y"], ["b", "d", "n"], ["b", "d", "n"]]
feature = pick_best_feature(data, features, attributes, label)
assert feature == "2"  # test 2 - picks attr with lower entropy


data = [["a", "c", "y"], ["b", "d", "n"]]
features = {"4", "5"}
feature = pick_best_feature(data, features, attributes, label)
assert feature is None  # test 3 - wrong features

<a id="domain"></a>
## domain

*`domain` Get the domain of a feature from a mapping of each feature to a tuple of its position in the dataset and a list of attributes in the domain of the feature. A list of values to consider Nan can be provided, by default attributes of "?" are skipped.* **Used by**: [id3](#id3)

* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (or label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **nans** List[str] - attributes to consider as missing values


**returns** List[str] - attributes in the domain of feature

In [27]:
def domain(attributes: Dict[str, Tuple[int, List[str]]], feature: str, nans: List[str] | None = None) -> List[str]:
    if nans is None:
        nans = ["?"]
    return [i for i in attributes[feature][1] if i not in nans]

In [28]:
attributes = {"1": (0, ["a", "b", "c"])}
assert domain(attributes, "1") == ["a", "b", "c"]  # test 1 - normal

attributes = {"1": (0, ["a", "b", "?"])}
assert domain(attributes, "1") == ["a", "b"]  # test 2 - remove missing


attributes = {"1": (0, ["?", "na"])}
assert domain(attributes, "1", ["?", "na"]) == []  # test 3 - returns empty list if all attributes are missing

<a id="subset_data"></a>
## subset_data

*`subset_data` Given a 2d list of data, a column index and a value, return all rows in the input data that have the value at the column index.* **Used by**: [id3](#id3)

* **data** List[List[str]] - a 2d list of observed attributes and label
* **column_idx** int - position in row to check for match 
* **value** str - value to filter data by


**returns** List[List[str]] - rows in data that have value at column index position

In [29]:
def subset_data(data: List[List[str]], column_idx: int, value: str) -> List[List[str]]:
    return [deepcopy(row) for row in data if row[column_idx] == value]

In [30]:
data = [["a", "y"], ["b", "y"], ["a", "n"]]
assert subset_data(data, 0, "a") == [["a", "y"], ["a", "n"]]  # test 1 - get rows

result = subset_data(data, 0, "a")
result[0][0] = "A"
assert data[0][0] == "a"  # test 2 - deepcopy

assert subset_data(data, 0, "c") == []  # test 3 - empty list if no match

<a id="remove_feature"></a>
## remove_feature

*`remove_feature` Remove feature from a set of features.* **Used by**: [id3](#id3)

* **features** Set[str] - a set of features
* **feature** str - feature to remove


**returns** Set[str] - set of features with feature removed

In [31]:
def remove_feature(features: Set[str], feature: str) -> Set[str]:
    return {f for f in features if f != feature}

In [32]:
features = {"1", "2", "3"}
assert remove_feature(features, "3") == {"1", "2"}  # test 1 - remove feature

features = {"1", "2"}
result = remove_feature(features, "1")
assert features == {"1", "2"}  # test 2 - doesnt mutate input


assert remove_feature({"1"}, "1") == set()  # test 3 - remove last key returns empty set

<a id="get_leaf_node"></a>
## get_leaf_node

*`get_leaf_node` Evaluate base cases for ID3 algorithm and return a leaf node if one is hit. If the data is empty, we return a leaf node with the value equal to the default label. If the data is homogenous, that is, all rows have the same label value, a leaf node with the value set to the homogenous label is returned. If the feature set is empty, a leaf node with the majority label of the data is returned. If no base cases are hit None is returned.* **Uses** [create_node](#create_node), [get_dominated_strategies](#get_homogeneous_label) and [get_majority_label](#get_majority_label) **Used by**: [id3](#id3)

* **data** List[List[str]] - a 2d list of observed attributes and label
* **features** Set[str] - set of features
* **label_idx** int - column index of label in data set
* **default_label** str - default leaf node value if data is empty 
* **trace** bool - if True print debug information

**returns** Node | None - a leaf node if a base case was hit, else None

In [33]:
def get_leaf_node(data: List[List[str]], features: Set[str], label_index: int, default_label: str, trace: bool = False) -> Node | None:
    if len(data) == 0:
        print("Base Case - empty data") if trace else None
        return create_node(default_label)

    homogenous_label = get_homogeneous_label(data, label_index)
    if homogenous_label is not None:
        print("Base Case - homogenous data") if trace else None
        return create_node(homogenous_label)

    if len(features) == 0:
        print("Base Case - empty features") if trace else None
        majority_label = get_majority_label(data, label_index)
        return create_node(majority_label)  # type: ignore - we already tested if data is empty
    return None

In [34]:
tree = get_leaf_node([], {"1"}, 1, default_label="y")
assert tree is not None and tree.value == "y"  # test 1 - empty data uses default label

data = [["a", "n"], ["b", "n"]]
tree = get_leaf_node(data, {"1", "2"}, 1, "n")
assert tree is not None and tree.value == "n"  # test 2 - homogeneous data

data = [["a", "y"], ["b", "y"], ["c", "n"]]
tree = get_leaf_node(data, set(), 1, "n")
assert tree is not None and tree.value == "y"  # test 3 - empty feature list returns majority

assert get_leaf_node(data, {"1"}, 1, "n") is None  # test 4 - no base cases hit return None

<a id="id3"></a>
## id3

*`id3` Given a 2d list of labeled observations, a set of features (columns) and a map of each feature/label to its column index and a list of attributes in its domain, create a decision tree to classify future observations. The tree is built recursively, at each step the feature with the lowest weighted entropy is selected and removed from the set of all features. The dataset is subdivided by the different attributes in the domain of the feature, and a sub-tree is made by running id3 on the subset. The sub-tree is then added as a child. If the attribute creates an empty subset, a leaf node with a default label is added to the tree. If the attribute perfectly subsets the data, so that each subset only contains a single label, leaf nodes are added to the tree with that label. When features is emptied, leaf nodes are added to the tree with the majority label of the subset. At each stage of the recursion, a new default label is generated by finding the majority label in the dataset. Each node in the tree contains two attributes, the first is a "value", which is the lowest entropy feature split on for an internal node. or the label value for a leaf node. The second attribute, "children" is a list of tuples, each containing the attribute being partitioned by and the child node. If a feature or majority label cannot be selected from the data an exception is raised.* **Uses** [get_leaf_node](#get_leaf_node), [pick_best_feature](#pick_best_feature), [get_majority_label](#get_majority_label), [create_node](#create_node), [subset_data](#subset_data), [remove_feature](#remove_feature), and [add_child](#add_child) **Used by**: [test](#test)

* **data** List[List[str]] - a 2d list of observed attributes and label
* **features** Set[str] - set of features
* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (or label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **default_label** str | None - default leaf node value if data is empty, if None the majority label value is used
* **label** str - label name
* **trace** bool - if True print debug information

**returns** Node - root of decision tree

In [35]:
def id3(data: List[List[str]], features: Set[str], attributes: Dict[str, Tuple[int, List[str]]], label: str, default_label: str, trace: bool = False) -> Node:
    print(f"{features=}, {attributes=}") if trace else None
    result = get_leaf_node(data, features, attributes[label][0], default_label, trace)
    if result is not None:
        return result

    feature = pick_best_feature(data, features, attributes, label, trace)
    if feature is None:
        raise Exception(f"Cannot find a feature to partition")

    majority_label = get_majority_label(data, attributes[label][0], trace)
    if majority_label is None:
        raise Exception(f"Cannot find a majority label")
    
    node = create_node(feature)
    for value in domain(attributes, feature):
        subset = subset_data(data, attributes[feature][0], value)
        child = id3(subset, remove_feature(features, feature), attributes, label, majority_label, trace)
        node = add_child(node, child, value)
    return node

In [36]:
data = [["a", "y"], ["b", "y"], ["a", "n"]]
attributes = {"1": (0, ["a", "b"]), "2": (1, ["y", "n"])}
features = {"1"}
tree = id3(data, features, attributes, "2", "y")

assert tree.value == "1"  # test 1.a - root is "1"
assert tree.children == [("a", Node(value = "y", children=[])), ("b", Node(value = "y", children=[]))] # test 1.b - children

data = [["a", "a", "y"], ["b", "a", "y"], ["a", "b", "n"]]
attributes = {"1": (0, ["a", "b"]), "2": (1, ["y", "n"])}
labels = ["y", "n"]
tree = id3(data, set(), attributes, "2", "y")
assert tree.value == "a" and tree.children == [] # test 3 - empty features returns base case


data = []
attributes = {"1": (0, ["a", "b"]), "2": (1, ["y", "n"])}
tree = id3(data, {"1"}, attributes, "2", "n" )
assert tree.value == "n" and tree.children == [] # test 3 - empty data returns default from base case

# Model 

<a id="train"></a>
## train

*`train` train decision tree using id3 algorithm on data.* **Uses** [get_majority_label](#get_majority_label) and [id3](#id3)

* **data** List[List[str]] - a 2d list of observed attributes and label
* **features** Set[str] - set of features
* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (or label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **label** str - label name
* **trace** bool - if True print debug information

**returns** Node - root of decision tree

In [37]:
def train(data: List[List[str]], features: Set[str], attributes: Dict[str, Tuple[int, List[str]]], label: str, trace=False) -> Node | None:
    default_label = get_majority_label(data, attributes[label][0], trace)
    if default_label is None:
        raise Exception("default label is None")
    return id3(data, features, attributes, label, default_label, trace)

<a id="classify"></a>
## classify

*`classify` Given a decision tree and a 2d list of observed attributes, estimate the label of each observation.* **Uses** [traverse_tree](#traverse_tree) 

* **tree** Node - root node of trained decision tree
* **observations** List[List[str]] - a 2d list of observed attributes
* **feature_indices** Dict[str, int] - mapping of feature to column index in data

**returns** List[str] - list of estimated labels for each row in observations

In [38]:
def classify(tree: Node, observations: List[List[str]], feature_indices: Dict[str, int]) -> List[str]:
    classifications = []
    for row in observations:
        label = traverse_tree(tree, row, feature_indices)
        classifications.append(label)
    return classifications

<a id="evaluate"></a>
## evaluate

*`evaluate` Given a list of true labels and a list of estimated labels, count the number of errors and update a confusion matrix by adding the number of TP, TN, FP, FN to the matrix.* **Uses** [traverse_tree](#traverse_tree) 

* **truth_set** List[str] - list of true labels
* **classifications** List[str] - list of estimated labels
* **confusion_matrix** Dict[str, int] - counter for TP, TN, FP, FN estimates

**returns** int - number of non-matching estimates

In [39]:
def evaluate(truth_set: List[str], classifications: List[str], labels: List[str], confusion_matrix: Dict[str, int]) -> int:
    errors = 0
    for true_label, estimate in zip(truth_set, classifications):
        if true_label == estimate and estimate == labels[1]:
            confusion_matrix["TP"] += 1

        elif true_label == estimate and estimate == labels[0]:
            confusion_matrix["TN"] += 1

        elif true_label != estimate and true_label == labels[0]:
            confusion_matrix["FP"] += 1
            errors += 1

        elif true_label != estimate and true_label == labels[1]:
            confusion_matrix["FN"] += 1
            errors += 1

        else:
            raise Exception(f"Unable to classify estimate: {estimate}")
    return errors

<a id="divide_folds"></a>
## divide_folds

*`divide_folds` Shuffle a dataset and divide into n leave-one-out training and test pairs.* **Uses** [create_folds](#create_folds) 

* **data** List[List[str]] - 2d list of observations and label
* **n_folds** int - number of folds to generate

**returns** List[Tuple[List[List[str]], List[List[str]]]] - List of tuple containing a training set and test set for each fold

In [40]:
def divide_folds(data: List[List[str]], n_folds: int = 10) -> List[Tuple[List[List[str]], List[List[str]]]]:
    random.shuffle(data)
    folds = create_folds(data, n_folds)

    k_folds = []
    for idx, test_fold in enumerate(folds):
        training_set = []
        for idx2, train_fold in enumerate(folds):
            if idx == idx2:
                continue
            training_set.extend(train_fold)

        k_folds.append((training_set, test_fold))
    return k_folds

<a id="cross_validate"></a>
## cross_validate

*`cross_validate` perform n_fold cross validation. For each fold, a train_fn is used to produce a model from the training data in the fold. The model is used to classify the test data in the fold using classify_fn. The classifcation is evaluated using the evaluate_fn. The error rate and a confusion matrix built from all the folds is returned. 

* **data** List[List[str]] - a 2d list of observed attributes and label
* **features** Set[str] - set of features
* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (or label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **label** str - label name
* **train_fn** Callable - function to produce model from training data
* **classify_fn** Callable - function to generate label estimates on test data using a model
* **evaluate_fn** Callable - function to collect the number of errors from the classification, as well as, update a confusion matrix.
* **n_folds** int - number of folds to use
* **trace** bool - if True print debug information

**returns** Tuple[float, Dict[str, int]] -  error rate and confusion matrix from all folds

In [41]:
def cross_validate(
    data: List[List[str]], features: Set[str], attributes: Dict[str, Tuple[int, List[str]]], label: str, train_fn: Callable = train, classify_fn: Callable = classify, eval_fn: Callable = evaluate, n_folds: int = 10, trace: bool = False
) -> Tuple[float, Dict[str, int]]:
    confusion_matrix: Dict[str, int] = {"TN": 0, "TP": 0, "FN": 0, "FP": 0}
    total_errors = 0
    feature_indices = {feature: idx for feature, (idx, _) in attributes.items()}
    label_idx, label_values = attributes[label]
    
    for training_set, test_set in divide_folds(data, n_folds):
        tree = train_fn(training_set, features, attributes, label, trace)
        classifications = classify_fn(tree, test_set, feature_indices)
        truth_set = [row[label_idx] for row in test_set]
        errors = eval_fn(truth_set, classifications, label_values, confusion_matrix)
        total_errors += errors

    error_rate = total_errors / len(data) if len(data) else 0.0
    return error_rate, confusion_matrix

# Run

<a id="run_model"></a>
## run_model

*`run_model` perform 10-fold cross validation on a dataset and then print the decision tree trained on the entire dataset.* **Uses** [split_features](#split_features), [cross_validate](#cross_validate), [train](#train) and [pretty_print_tree](#pretty_print_tree)

* **data** List[List[str]] - a 2d list of observed attributes and label
* **features** Set[str] - set of features
* **attributes** Dict[str, Tuple[int, List[str]]] - a map of each feature (or label) to a tuple of its position in the dataset and a list of attributes in the domain of the feature.
* **label** str - label name
* **trace** bool - if True print debug information

**returns** 

In [42]:
def run_model(data: List[List[str]], attributes: Dict[str, Tuple[int, List[str]]], label: str, trace: bool = False):
    n_folds = 10

    features = split_features(attributes, label)

    avrg_error_rate, cm = cross_validate(data, features, attributes, label, n_folds=n_folds)
    print(f"\nConfusion Matrix ({n_folds}-fold CV):")
    print(f"TP={cm['TP']}  FP={cm['FP']}\nFN={cm['FN']}  TN={cm['TN']}")
    print(f"\nAverage Error Rate ({n_folds}-fold CV): {avrg_error_rate:0.4f}")

    tree = train(data, features, attributes, label, trace=trace)
    assert tree is not None

    print("\nDecision Tree:")
    pretty_print_tree(tree)

In [43]:
attributes = {"Shape": (0, ["round", "square"]), "Size": (1, ["large", "small"]), "Color": (2, ["blue", "green", "red"]), "Safe?": (3, ["yes", "no"])}
label = "Safe?"

data = [
    ["round", "large", "blue", "no"],
    ["square", "large", "green", "yes"],
    ["square", "small", "red", "no"],
    ["round", "large", "red", "yes"],
    ["square", "small", "blue", "no"],
    ["round", "small", "blue", "no"],
    ["round", "small", "red", "yes"],
    ["square", "small", "green", "no"],
    ["round", "large", "green", "yes"],
    ["square", "large", "green", "yes"],
    ["square", "large", "red", "no"],
    ["square", "large", "green", "yes"],
    ["round", "large", "red", "yes"],
    ["square", "small", "red", "no"],
    ["round", "small", "green", "no"],
]

run_model(data, attributes, label)


Confusion Matrix (10-fold CV):
TP=6  FP=1
FN=2  TN=6

Average Error Rate (10-fold CV): 0.2000

Decision Tree:
 - Size
     | large
         - Color
             | blue -> no
             | green -> yes
             | red
                 - Shape
                     | round -> yes
                     | square -> no
     | small
         - Color
             | blue -> no
             | green -> no
             | red
                 - Shape
                     | round -> yes
                     | square -> no


In [44]:
trace = False
data = parse_data("./agaricus-lepiota-3.data")
attributes, abrv2name = parse_attributes("./agaricus-lepiota-3.attrs.json")
label = "mushroom-type"

data = rename_data(data, attributes, abrv2name)
assert data is not None

run_model(data, attributes, label)


Confusion Matrix (10-fold CV):
TP=4208  FP=0
FN=0  TN=3916

Average Error Rate (10-fold CV): 0.0000

Decision Tree:
 - odor
     | almond -> edible
     | anise -> edible
     | creosote -> poisonous
     | fishy -> poisonous
     | foul -> poisonous
     | musty -> poisonous
     | none
         - spore-print-color
             | black -> edible
             | brown -> edible
             | buff -> edible
             | chocolate -> edible
             | green -> poisonous
             | orange -> edible
             | purple -> edible
             | white
                 - habitat
                     | grasses -> edible
                     | leaves
                         - cap-color
                             | brown -> edible
                             | buff -> edible
                             | cinnamon -> edible
                             | gray -> edible
                             | green -> edible
                             | pink -> edible
                  

## Before You Submit...

1. Did you provide output exactly as requested?
2. Did you re-execute the entire notebook? ("Restart Kernel and Rull All Cells...")
3. If you did not complete the assignment or had difficulty please explain what gave you the most difficulty in the Markdown cell below.
4. Did you change the name of the file to `jhed_id.ipynb`?

Do not submit any other files.